# 第 03 步：累计收益与理论持有净值

<!-- SNAPSHOT_NOTE -->

## 数据准备

本页记录的是完整快照上的实际运行。从 GitHub 克隆源码后，先在仓库根目录运行 `python publication/restore_snapshot.py`，恢复同版本的 `data/` 与 `runs/`，再执行本步。[完整数据与大型结果索引](../../publication/ASSET_INDEX.md)保留全部路径和哈希；本页图表与 Notebook 已保存输出可直接阅读。

## 目标与范围

得到 `SYN001` 的简单日收益后，本步要回答：从当前快照的第一天收盘开始持续持有，忽略费用、没有追加或提取资金，财富会怎样变化？本步建立的是单只人工股票的理论持有序列，不是根据未来结果选出的交易策略，也不是后续月度多因子组合的净值。

沿用 [第 02 步](../02_daily_returns/README.md) 的同一份行情与日历，在仓库根目录执行：

```powershell
.\.venv\Scripts\python.exe examples/03_cumulative_returns.py
```

输入仍是 `runs/first_demo_20261107/input/prices.csv`。脚本在内存中计算并打印，不改写原始输入。计算前先核对单股价格完整且为正；若发现缺失，应返回数据检查，不能为了连乘方便而自动填价。

## 从财富递推推导复利

设起点财富为 `W_0`，第 t 个区间的简单收益为 `r_t`。经过一期，财富变成 `W_1=W_0(1+r_1)`；第二期的收益发生在已经变化的财富上，因此 `W_2=W_0(1+r_1)(1+r_2)`。推广后：

$$
W_T=W_0\prod_{t=1}^{T}(1+r_t),\qquad
NAV_T=\frac{W_T}{W_0},\qquad
R_{0,T}=NAV_T-1.
$$

`1+r` 表示财富增长倍数。例如 +3% 对应 1.03，-3% 对应 0.97。净值 1 表示财富等于起点，净值 1.10 表示累计增长 10%，净值 0.80 表示累计减少 20%。净值本身不是百分数，也不是每期收益率。

一个用于理解公式的算术例子是先 +10%、再 -10%：财富倍数为 `1.1×0.9=0.99`，累计收益 -1%，并非 0%。这是解释复利的例子，不对应当前行情中的两个实际交易日。

## 起点设定与连乘代码

```python
stock["daily_return"] = (
    stock["adj_close"] / stock["adj_close"].shift(1) - 1
)
growth_factor = 1 + stock["daily_return"]
growth_factor.iloc[0] = 1.0
stock["nav"] = growth_factor.cumprod(skipna=False)
stock["cumulative_return"] = stock["nav"] - 1
```

第一段重新计算小数日收益，定义与上一节一致。第二行把收益转换为增长倍数。第三行只设定连乘起点：从首日收盘开始计量净值，起点倍数设为 1。这里没有把 `daily_return.iloc[0]` 改成 0，首日相对于更早未知价格的收益仍是缺失。

`cumprod` 按顺序累计相乘，`skipna=False` 明确不在中途遇到未知值后跳过它继续拼接财富。当前脚本先拒绝空表、缺失和非正收盘价，使这个交叉核对例子建立在完整价格上；若未来换成其他输入，仍需要更全面的价格与来源校验。

最后一行从净值减去起点 1，得到累计收益率。这个公式依赖单位净值的约定。如果净值起点设为 100，累计收益就应为 `NAV/100-1`，不能再直接减 1。

## 从净值换算财富金额

```python
initial_capital = 10_000.0
stock["wealth"] = initial_capital * stock["nav"]
stock["wrong_sum_of_returns"] = stock["daily_return"].cumsum()
```

这里将初始资金设为 10,000 元，是为了把无量纲净值转成容易理解的财富金额。`wealth` 与 `nav` 完全按同一比例变化，换成其他初始金额不会改变收益率或最大回撤。这个简单例子没有现金流、税费、整手和可成交约束，不能替代真实账户账本。

`wrong_sum_of_returns` 是特意保留的反例列，累计的是每天的简单收益率之和，而不是财富增长，不能当作另一种正确净值。保留这个对照，可以在实际数据上看清错误的后果，而不只记住“收益要复利”这句话。

## 前几行实际结果

| 日期 | 日收益 | 净值 | 累计收益 | 10,000 元理论财富 |
| --- | ---: | ---: | ---: | ---: |
| 2019-01-01 | NaN | 1.0000000000 | 0.000000% | 10,000.0000 |
| 2019-01-02 | -2.93633667% | 0.9706366333 | -2.936337% | 9,706.3663 |
| 2019-01-03 | 3.35370402% | 1.0031889131 | 0.318891% | 10,031.8891 |
| 2019-01-04 | 0.24360084% | 1.0056326897 | 0.563269% | 10,056.3269 |
| 2019-01-07 | -1.58498160% | 0.9896935967 | -1.030640% | 9,896.9360 |
| 2019-01-08 | 1.68768520% | 1.0063965090 | 0.639651% | 10,063.9651 |

重点手算首日之后的前两个收益区间：

```text
(1 - 0.02936336673611) × (1 + 0.03353704024677) - 1
≈ 0.00318891309865
≈ 0.31889131%
```

如果直接相加，得到 `0.0041736735106571`，约 **0.41736735%**。两者相差约 **0.09847604 个百分点**，原因是第二期收益作用于第一期变化后的本金。百分比与百分点也不同：两个收益率的差值使用“百分点”表达，不能将其直接解释为另一种相对增长。

## 用价格比独立核对

```python
nav_from_price = stock["adj_close"] / stock["adj_close"].iloc[0]
max_difference = (stock["nav"] - nav_from_price).abs().max()
```

从 `1+r_t=P_t/P_(t-1)` 出发，把所有倍数连乘，中间价格应当逐项抵消，最后只剩 `P_T/P_0`。因此，在完整、统一复权口径、没有外部现金流的当前例子中，连乘净值应与价格比一致。

当前最大差异约 **2.4424906541753444×10^-15**，属于浮点运算量级。不必通过四舍五入强行消除这种尾数，也不能据此认为两份数据来源独立；两种算法依然使用同一价格输入。这个核对主要检查连乘方向、起点与公式是否正确。

另外，`daily_return` 第一行仍保留缺失值。起点净值为 1、累计收益为 0，只是从该点开始计量的定义，不会补出首日前未知的价格变化。

## 实际期末记录

| 指标 | 当前固定样本结果 |
| --- | ---: |
| 结束日期 | 2025-12-31 |
| 首日复权收盘价 | 26.327305296402876 |
| 期末复权收盘价 | 9.123247365115686 |
| 期末净值 | 0.3465317571396949 |
| 累计收益率 | -65.3468242860% |
| 10,000 元理论财富 | 3,465.317571396949 |
| 错误的日收益直接相加 | -63.9760601822% |

我保留这份负收益结果，没有为了展示而更换股票或截短区间。它只是一个人工单股持有例子，并不能支持或否定多因子策略，更不能推出真实 A 股的长期表现。结果不理想时，公式和数据核对仍然需要完整完成。

## 价格比核对成立的条件

连乘等于价格比，是因为相邻比值使用同一条连续价格序列，而且中间没有另加账户现金流。这不是任意投资账户都适用的捷径。中途追加资金会使账户金额上升，但增加的金额不全是投资回报；卖出一部分持仓并持有现金后，账户也不再等于某只股票价格按固定倍数缩放。后续组合回测必须逐日维护现金和实际持仓，而不是继续拿一个价格比代替。

本例假设首日收盘建立估值起点，目的在于解释复利，不是在主张真实订单能以该价格无成本成交。后续策略使用收盘数据形成信号，执行起点改为下一交易日开盘，初始资金、首次成本和当日收盘估值也会进入账户。因此不同章节的曲线即使使用相同股票，也需要先对齐起点和假设才能比较。

简单收益与对数收益还需要分开讨论其可加性。对于正价格，若定义对数收益为相邻价格比的自然对数，它们可以通过加总还原总的对数价格变化；但那是另一个定义，恢复简单累计收益还需要指数变换。当前列明确是简单收益，所以不能借用对数收益的可加性来为直接相加辩解。本节保持原定义，用连乘和价格比完成核对，不在中途更换收益口径。

最后检查数据展示和计算精度。表格中保留六位或十位小数是为了可读性；若先把每天收益四舍五入再反复连乘，长期结果可能与原始价格比产生额外差异。数值核对在原始浮点精度上进行，展示舍入放在最后。数值误差很小只说明这两种代数计算对得上，不代表人工样本具有真实市场代表性。

## 常见错误

| 错误操作 | 为什么会错 |
| --- | --- |
| 用 `cumsum` 计算简单收益的累计回报 | 忽略每一期变化后的财富基数 |
| 把百分数列直接传给 `1+r` | 收益被放大 100 倍 |
| 把第一天未知收益改成 0 | 把起点约定伪装成已观测收益 |
| 让连乘静默跳过中间缺失 | 财富路径绕过未知变化，失去可解释性 |
| 混用财富金额与净值 | 金额有货币单位，净值是相对起点的倍数 |
| 把单股理论持有称为可实现策略 | 尚未模拟成交、现金、费用和持仓约束 |

## 验收记录与下一步

本步验收包括：从一期财富递推推导连乘公式，手算前两个收益区间，说明首日净值与首日收益的区别，并用价格比核对整个序列。当前结论仅限于这份完整人工价格序列：连乘与价格比一致，直接加总简单收益不等价。

下一步进入 [第 04 步：净值与最大回撤](../04_nav_drawdown/README.md)。期末收益只有一个终点数字，还需要观察到达终点之前，财富曾经从高点跌落多少。

## 完整可运行实现

上面的片段用于解释思路；下面各单元按顺序执行完整脚本。先定位仓库根目录，并显式设置脚本路径，使 `__file__` 在 Notebook 中也有明确含义。

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "pyproject.toml").exists()
                     and (candidate / "examples/01_read_prices.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("我需要从本项目目录或它的子目录打开 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
__file__ = str(PROJECT_ROOT / 'examples/03_cumulative_returns.py')
sys.argv = [__file__]


### 完整实现 · 单元 1

In [2]:
"""第三课：复利累计收益与单只模拟股票的理论持有净值。"""

from pathlib import Path
import pandas as pd

# 沿用前两课：读取数据、筛选一只股票、对齐模拟交易日历。
project_dir = Path(__file__).resolve().parents[1]
data_path = project_dir / "runs" / "first_demo_20261107" / "input" / "prices.csv"
prices = pd.read_csv(data_path, dtype={"symbol": "string"}, parse_dates=["date"])
symbol = "SYN001"
stock = prices.loc[prices["symbol"] == symbol].copy()
stock = stock.sort_values("date").reset_index(drop=True)
calendar = pd.read_csv(data_path.with_name("calendar.csv"), parse_dates=["date"])
stock = stock.set_index("date").reindex(calendar["date"]).reset_index()

### 完整实现 · 单元 2

In [3]:
# 本课交叉核对需要完整价格；真实缺失行情应先调查，不能自动补价。
if stock.empty or stock["adj_close"].isna().any() or (stock["adj_close"] <= 0).any():
    raise ValueError("本课需要完整且为正的复权收盘价，请先检查股票代码和缺失行情。")

### 完整实现 · 单元 3

In [4]:
# 1. 日收益率；第一天没有前一日价格，所以仍然保留 NaN。
stock["daily_return"] = stock["adj_close"] / stock["adj_close"].shift(1) - 1

# 2. 把日收益率转换为财富增长倍数，例如 +3% 对应 1.03。
growth_factor = 1 + stock["daily_return"]

# 3. 以第一天收盘为估值起点，起点净值设为 1。
# 这里只设置连乘起点，没有把第一天的未知日收益改成 0。
growth_factor.iloc[0] = 1.0

# 4. 按时间顺序连乘。若中途出现未知值，不跳过它继续计算。
stock["nav"] = growth_factor.cumprod(skipna=False)

# 5. 净值减去初始的 1，就是从起点至当天的累计收益率。
stock["cumulative_return"] = stock["nav"] - 1

# 6. 假设初始资金为 1 万元，后续没有追加或取出资金。
# 这是连续持有这只股票、忽略费用的理论财富序列。
initial_capital = 10_000.0
stock["wealth"] = initial_capital * stock["nav"]

# 7. 仅作反例：把每天的简单收益率直接相加，会得到不同结果。
stock["wrong_sum_of_returns"] = stock["daily_return"].cumsum()

columns = ["date", "adj_close", "daily_return", "nav", "cumulative_return", "wealth"]
print("模拟股票：", symbol)
print("起点为第一天收盘，nav = 1；收益率列保存小数。")
print(stock[columns].head(6).to_string(index=False))

print("\n只看起点后的前两个收益区间：")
row = stock.iloc[2]
print(f"日期：{row['date'].date()}")
print(f"正确累计收益：{row['cumulative_return']:.6%}")
print(f"错误的直接相加：{row['wrong_sum_of_returns']:.6%}")

# 8. 完整、统一复权口径下，净值还应等于当日价格 / 起点价格。
nav_from_price = stock["adj_close"] / stock["adj_close"].iloc[0]
max_difference = (stock["nav"] - nav_from_price).abs().max()
print("\n连乘净值与价格比净值的最大差异：", max_difference)
print("第一天的日收益仍为缺失值：", bool(pd.isna(stock["daily_return"].iloc[0])))
print("\n期末结果：")
print(stock[columns].tail(1).to_string(index=False))

模拟股票： SYN001
起点为第一天收盘，nav = 1；收益率列保存小数。
      date  adj_close  daily_return      nav  cumulative_return       wealth
2019-01-01  26.327305           NaN 1.000000           0.000000 10000.000000
2019-01-02  25.554247     -0.029363 0.970637          -0.029363  9706.366333
2019-01-03  26.411261      0.033537 1.003189           0.003189 10031.889131
2019-01-04  26.475599      0.002436 1.005633           0.005633 10056.326897
2019-01-07  26.055965     -0.015850 0.989694          -0.010306  9896.935967
2019-01-08  26.495708      0.016877 1.006397           0.006397 10063.965090

只看起点后的前两个收益区间：
日期：2019-01-03
正确累计收益：0.318891%
错误的直接相加：0.417367%

连乘净值与价格比净值的最大差异： 2.4424906541753444e-15
第一天的日收益仍为缺失值： True

期末结果：
      date  adj_close  daily_return      nav  cumulative_return      wealth
2025-12-31   9.123247      0.015874 0.346532          -0.653468 3465.317571


## 运行复核

以上实现已在干净内核中按顺序执行；执行状态记录在仓库的 [Notebook 校验记录](../../publication/notebook_validation.json)。代码中的数据身份、费用和成交假设仍然适用，运行成功只验证对应实现。